## Proyecto SQL Sprint 14

### 1. Introducción 

Durante el coronavirus aparecieron empresas emergentes que se desarrollaron a crear nuevas aplicaciones para los amantes de los libros, debido a que la gente ya no podía salir a pasar su tiempo libre fuera, por lo que se quedaban en casa leyendo más libros. 

Se analizará datos sobre un servicio que compite en este mercado para generar una propuesta de valor para un nuevo producto. Se estudiará información sobre los libros como editoriales, autores, calificaciones y reseñas. 

El objetivo es identificar los libros con mayor cantidad de reseñas y mejores calificaciones. La editorial con mayor cantidad de libros publicados, el autor con la calificación mas alta y el número promedio de reseñas de texto entre usuarios que han calificado más de 50 libros. 

#### 2. Importar librerías y conectarse a la base de datos

In [29]:
# importar librerías 
import pandas as pd
from sqlalchemy import create_engine

In [30]:
#conectarse a la base de datos 
db_config = {'user': 'practicum_student',         # nombre de usuario
             'pwd': 's65BlTKV3faNIGhmvJVzOqhs', # contraseña
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432,              # puerto de conexión
             'db': 'data-analyst-final-project-db'}          # nombre de la base de datos

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                                     db_config['pwd'],
                                                                       db_config['host'],
                                                                       db_config['port'],
                                                                       db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [31]:
#imprimir las primeras filas de las tablas 
def primeras_filas (table, engine, n = 5):
    query = f"SELECT * FROM {table} LIMIT {n};"
    df = pd.read_sql (query, engine)
    print (f"\n {table}:")
    print (df)

tables = ['books', 'authors', 'publishers', 'ratings', 'reviews']

for table in tables: 
    primeras_filas (table, engine)


 books:
   book_id  author_id                                              title  \
0        1        546                                       'Salem's Lot   
1        2        465                 1 000 Places to See Before You Die   
2        3        407  13 Little Blue Envelopes (Little Blue Envelope...   
3        4         82  1491: New Revelations of the Americas Before C...   
4        5        125                                               1776   

   num_pages publication_date  publisher_id  
0        594       2005-11-01            93  
1        992       2003-05-22           336  
2        322       2010-12-21           135  
3        541       2006-10-10           309  
4        386       2006-07-04           268  

 authors:
   author_id                          author
0          1                      A.S. Byatt
1          2  Aesop/Laura Harris/Laura Gibbs
2          3                 Agatha Christie
3          4                   Alan Brennert
4          5        Al

Todas las tablas estan conformadas por las variables correctas 

### 3. Ejercicios

#### 3.1. Encuentra el número de libros publicados después del 1 de enero de 2000.

In [32]:
query1 = """ 
SELECT COUNT (*) AS libros_despues_2000
FROM books 
WHERE publication_date > '2000-01-01'
"""

libros_despues_2000 = pd.read_sql (query1, engine)
print (libros_despues_2000)

   libros_despues_2000
0                  819


Después del 1 de enero del 2000 se han publicado 819 libros. 

#### 3.2 Encuentra el número de reseñas de usuarios y la calificación promedio para cada libro.

In [54]:
query2 = """ 
SELECT 
    books.book_id, 
    books.title, 
    COUNT (DISTINCT reviews.review_id) AS total_reviews, 
    AVG (ratings.rating) AS avg_rating
FROM 
    books
    LEFT JOIN reviews ON reviews.book_id  = books.book_id
    LEFT JOIN ratings ON ratings.book_id = books.book_id
GROUP BY 
    books.book_id, 
    books.title
ORDER BY
    avg_rating DESC,
    total_reviews DESC
"""


resena_calificacion = pd.read_sql (query2, engine)
print (resena_calificacion)


     book_id                                              title  \
0         17                      A Dirty Job (Grim Reaper  #1)   
1        553            School's Out—Forever (Maximum Ride  #2)   
2        444       Moneyball: The Art of Winning an Unfair Game   
3         86      Arrows of the Queen (Heralds of Valdemar  #1)   
4        972  Wherever You Go  There You Are: Mindfulness Me...   
..       ...                                                ...   
995      915  The World Is Flat: A Brief History of the Twen...   
996      202                                      Drowning Ruth   
997      316                  His Excellency: George Washington   
998      371                                              Junky   
999      303                               Harvesting the Heart   

     total_reviews  avg_rating  
0                4        5.00  
1                3        5.00  
2                3        5.00  
3                2        5.00  
4                2        5.00

Se obtienen el total de reseñas para cada libro y el promedio de las calificaciones para cada uno.
El libro con mayor puntaje y más reseñas para ese puntaje es A Dirty Job (Grim Reaper #1), seguido de  School's Out—Forever (Maximum Ride  #2) con 3 reseñas. 

In [34]:
#se verifica que hayan la cantidad de libros diferentes que salen en el resultado anterior.
query3 = """ 
SELECT COUNT (DISTINCT book_id) AS total
FROM books
"""

total = pd.read_sql (query3, engine)
print (total)

   total
0   1000


Se confirma que se tienen 1000 libro diferentes. 

In [35]:
#se calcula el máximo de reseñas que ha obtenido un libro y la calificación máxima para un libro 
query4 = """ 
SELECT 
    MAX(total_reviews) AS max_total_reviews,
    MAX(avg_rating) AS max_avg_rating
FROM (
    SELECT 
        COUNT(DISTINCT reviews.review_id) AS total_reviews,
        AVG(ratings.rating) AS avg_rating
    FROM 
        books
        LEFT JOIN reviews ON reviews.book_id = books.book_id
        LEFT JOIN ratings ON ratings.book_id = books.book_id
    GROUP BY 
        books.book_id
) AS subquery;
"""
max = pd.read_sql (query4, engine)
print (max)


   max_total_reviews  max_avg_rating
0                  7             5.0


El máximo de reseñas que ha tenido un libro es de 7 usarios y la máxima calificación que ha resivido un libro es de 5. 

#### 3.3 Identifica la editorial que ha publicado el mayor número de libros con más de 50 páginas (esto te ayudará a excluir folletos y publicaciones similares de tu análisis).

In [41]:
query5 = """ 
SELECT 
    publishers.publisher AS name_publisher,
    COUNT (books.book_id) AS total_books
FROM 
books 
    LEFT JOIN publishers ON publishers.publisher_id = books.publisher_id
WHERE
    num_pages > '50'
GROUP BY
    name_publisher
ORDER BY
    total_books DESC
LIMIT 5
"""
mayorpublicaciones= pd.read_sql (query5, engine)
print (mayorpublicaciones)


             name_publisher  total_books
0             Penguin Books           42
1                   Vintage           31
2  Grand Central Publishing           25
3          Penguin Classics           24
4                    Bantam           19


La editorial que con mayor libros publicados es Peguin Books con 42 libros de más de 50 páginas. 

#### 3.4 Identifica al autor que tiene la más alta calificación promedio del libro: mira solo los libros con al menos 50 calificaciones.

In [45]:
query6 = """ 
SELECT 
    authors.author AS author_name, 
    AVG (ratings.rating) AS avg_rating
FROM 
books 
    JOIN authors ON authors.author_id = books.author_id
    JOIN ratings ON ratings.book_id = books.book_id
GROUP BY
    author_name
HAVING 
    COUNT (ratings.book_id) >= 50
ORDER BY
    avg_rating DESC
LIMIT 5
"""
mejorautor= pd.read_sql (query6, engine)
print (mejorautor)


                         author_name  avg_rating
0                     Diana Gabaldon    4.300000
1         J.K. Rowling/Mary GrandPré    4.288462
2                    Agatha Christie    4.283019
3  Markus Zusak/Cao Xuân Việt Khương    4.264151
4                     J.R.R. Tolkien    4.240964


El autor que tienen la calificación promedio del libro más alta es Diana Gabaldon con una claificación de 4.3

#### 3.5 Encuentra el número promedio de reseñas de texto entre los usuarios que calificaron más de 50 libros.

Primero se crea una tabla donde se almacena el resultado del conteo de libros para después poder sacar el promedio de esta columna. 
Después se hace una subcolsuta donde se filtran los usuarios que han calificado a mas de 50 libros de la tabla ratings para que solo esos se los considere para calcular la cantidad de reseñas de texto. 
Finalmente se saca el promedio del total de reseñas de los usuarios ya filtrados. 

In [49]:
query7 = """ 
WITH user_reviews_count AS (
    SELECT 
        username AS user_reviews, 
        COUNT (DISTINCT review_id) AS total_reviews 
    FROM 
        reviews
    WHERE 
        username IN (
        SELECT username 
        FROM ratings
        GROUP BY username 
        HAVING COUNT (book_id) > 50
    )
    GROUP BY
        user_reviews
)
SELECT
    AVG (total_reviews) AS avg_total_reviews_by_user
FROM
    user_reviews_count
"""
avg_resenas= pd.read_sql (query7, engine)
print (avg_resenas)

   avg_total_reviews_by_user
0                  24.333333


EL promedio de reseñas de texto entre los usuarios que calificaron a más de 50 libros es de 24.33 reseñas. 

### 4. Conclusiones 

Resumiendo las conslusiones obtenidas para cada consulta: 

    - 1. La cantidad de libros publicados después del 2000 fue del 819. 

    - 2. Se identifica el número de reseñas y la calificación promedio que los usuarios han dado a cada libro donde se determina que el máximo de reseñas que ha obtenido un libro es de 7 usuarios y la calificación más alta para un libro es de 5 puntos. El libro con mayor puntuación y mayor número de reseñas para esa puntuación es  A Dirty Job (Grim Reaper #1).

    - 3. La editorial que más libros ha publicado es Penguien Books con 42 libros que tengan más de 50 páginas. 

    - 4. La autora Diana Gabaldon es la que obtuvo la mayor calificación promedio con 4.30 considerando solo aquellos autores que han tenido más de 50 calificaciones. 

    - 5. El promedio de reseñas de texto entre usuarios que han calificado a más de 50 libros es de 24 reseñas aproximadamente.  

Al obtener estos resultados para las consultas, se puede recomendar que los libros que se incluyan en esta nueva aplicación sean aquellos que tengan una major calificación de los usuarios y a su vez con varias reseñas. 

También se debe considerar los libros de la editorial con mayor cantidad de libros publicados, siempre y cuando estos libros tengan una calificación razonable para poder incluirlos. 

Así mismo es necesario considerar los libros de la autora mejor calificada, ya que se puede generar una tendencia en cuanto a la lectura de los libros escritos por esta autora. 

Siempre es necesario tomar en cuanta las reseñas de texto y leerlas cuidadosamente para saber las inclinaciones y gustos de los usuarios que serían los potenciales clientes. 